In [52]:
import pandas as pd
import numpy as np
from datetime import datetime

In [53]:
dataset = pd.read_csv(r'../data/NepalLatestAQI.csv')
dataset.head()

,date,station,latitude,longitude,aqi,pm2_5,pm10,no2,so2,o3,temperature_C,relative_humidity_%,notes
0,2024-01-01,Kathmandu,27.7172,85.3240,162,75.9,161.5,18.5,8.0,16.2,20.9,65.2,NaN
1,2024-01-01,Lalitpur,27.6648,85.3188,157,66.9,105.9,18.2,4.2,34.1,28.8,76.2,NaN
2,2024-01-01,Bhaktapur,27.6714,85.4270,166,84.8,182.9,16.4,7.1,26.7,20.3,60.9,NaN
3,2024-01-01,Pokhara,28.2096,83.9856,79,25.5,47.8,2.0,1.0,31.7,23.0,47.9,NaN
4,2024-01-01,Biratnagar,26.4525,87.2718,146,53.8,86.3,17.2,1.9,34.5,23.9,54.1,NaN


In [54]:
dataset.isnull().sum()

date                      0
station                   0
latitude                  0
longitude                 0
aqi                       0
pm2_5                     0
pm10                      0
no2                       0
so2                       0
o3                        0
temperature_C             0
relative_humidity_%       0
notes                  9761
dtype: int64

In [55]:
dataset = dataset.drop(columns='notes')

In [56]:
dataset = dataset.rename(columns=lambda c: c.strip())

In [57]:
dataset = dataset.sort_values('date').drop_duplicates(subset=['station', 'date'], keep='last')

In [58]:
dataset['date'] = pd.to_datetime(dataset['date'])
dataset['month'] = dataset['date'].dt.month
dataset['day'] = dataset['date'].dt.day
dataset['dayofweek'] = dataset['date'].dt.dayofweek
dataset['is_weekend'] = dataset['dayofweek'].isin([5,6]).astype(int)

❗ Why do we encode cyclical features?

Months and weekdays are cyclical:

After December (12) comes January (1)

After Sunday (6) comes Monday (0)

If you use raw numbers:

The model thinks January (1) is far from December (12)
But they are next to each other.


In [59]:
dataset['month_sin'] = np.sin(2 * np.pi * dataset['month']/12)
dataset['month_cos'] = np.cos(2 * np.pi * dataset['month']/12)
dataset['dow_sin'] = np.sin(2 * np.pi * dataset['dayofweek']/7)
dataset['dow_cos'] = np.cos(2 * np.pi * dataset['dayofweek']/7)

In [60]:
dataset.columns

Index(['date', 'station', 'latitude', 'longitude', 'aqi', 'pm2_5', 'pm10',
       'no2', 'so2', 'o3', 'temperature_C', 'relative_humidity_%', 'month',
       'day', 'dayofweek', 'is_weekend', 'month_sin', 'month_cos', 'dow_sin',
       'dow_cos'],
      dtype='object')

In [61]:
cols = ['aqi', 'pm2_5', 'pm10', 'no2', 'so2', 'o3', 'temperature_C', 'relative_humidity_%']

In [62]:
lags = [1, 3, 7]
rolls = [3, 7]

In [63]:
dataset = dataset.sort_values(['station', 'date']).reset_index(drop=True)

# Lags & rolling use only past values (shifted), so no leakage occurs


In [64]:
# create lag feature
for c in cols:
    for lag in lags:
        new_col = f"{c}_lag{lag}"
        dataset[new_col] = dataset.groupby('station')[c].shift(lag)

In [65]:
# crete rolling-mean features
for c in cols:
    for w in rolls:
        new_col = f"{c}_roll{w}"
        dataset[new_col] = (dataset.groupby('station')[c].transform(lambda s: s.rolling(window=w, min_periods=1).mean().shift(1)))

In [66]:
new_features = [c for c in dataset.columns if any(x in c for x in ['_lag', '_roll'])]
print(f"Added {len(new_features)} features: {new_features[:20]}{'' if len(new_features)<=20 else ' ...'}")
print("\nMissing values count for new features:")
print(dataset[new_features].isnull().sum())

Added 40 features: ['aqi_lag1', 'aqi_lag3', 'aqi_lag7', 'pm2_5_lag1', 'pm2_5_lag3', 'pm2_5_lag7', 'pm10_lag1', 'pm10_lag3', 'pm10_lag7', 'no2_lag1', 'no2_lag3', 'no2_lag7', 'so2_lag1', 'so2_lag3', 'so2_lag7', 'o3_lag1', 'o3_lag3', 'o3_lag7', 'temperature_C_lag1', 'temperature_C_lag3'] ...

Missing values count for new features:
aqi_lag1                      15
aqi_lag3                      45
aqi_lag7                     105
pm2_5_lag1                    15
pm2_5_lag3                    45
pm2_5_lag7                   105
pm10_lag1                     15
pm10_lag3                     45
pm10_lag7                    105
no2_lag1                      15
no2_lag3                      45
no2_lag7                     105
so2_lag1                      15
so2_lag3                      45
so2_lag7                     105
o3_lag1                       15
o3_lag3                       45
o3_lag7                      105
temperature_C_lag1            15
temperature_C_lag3            45
temperatur

In [67]:
initial_len = len(dataset)
dataset = dataset[~dataset[new_features].isnull().any(axis=1)].reset_index(drop=True)
dropped = initial_len - len(dataset)
print(f"Dropped {dropped} rows ({dropped/initial_len*100:.2f}%) because lag/roll features were not available.")

Dropped 105 rows (1.07%) because lag/roll features were not available.


In [68]:
dataset['station'].nunique()

15

In [69]:
dataset = pd.get_dummies(dataset, columns=['station'], prefix='st', drop_first=False)

In [70]:
dataset.shape

(9735, 74)

In [71]:
train_end = '2025-05-30'
valid_end = '2025-08-30'

In [72]:
train_data = dataset[ dataset['date'] <= train_end ]
valid_data = dataset[ (dataset['date'] > train_end) & (dataset['date'] <= valid_end) ]
test_data = dataset[ dataset['date'] > valid_end ]

In [73]:
print("Train shape :", train_data.shape)
print("Valid shape :", valid_data.shape)
print("Test shape  :", test_data.shape)

Train shape : (7635, 74)
Valid shape : (1380, 74)
Test shape  : (720, 74)


In [74]:
target = 'aqi'
features = [c for c in dataset.columns if c not in ['date', 'aqi']]

In [75]:
x_train = train_data[features]
y_train = train_data[target]

x_valid = valid_data[features]
y_valid = valid_data[target]

x_test = test_data[features]
y_test = test_data[target]

In [76]:
from xgboost import XGBRegressor

In [77]:
model = XGBRegressor( n_estimators= 1844, 
learning_rate= 0.005138423447513474, 
max_depth= 6, 
subsample= 0.6011047426660859, 
colsample_bytree= 0.8897617488503717, 
reg_alpha= 0.0065837707426329595, 
reg_lambda= 8.5800788151842, 
min_child_weight= 2, 
gamma= 1.4098179177734331,
tree_method="hist", 
verbosity=0,
early_stopping_rounds=50)

model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8897617488503717
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,None


In [78]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [79]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    return {'mae': mae, 'rmse': rmse, 'mape': mape}

In [80]:
pred_valid = model.predict(x_valid)
pred_test = model.predict(x_test)

In [81]:
metrics_valid = compute_metrics(y_valid, pred_valid)
metrics_test  = compute_metrics(y_test, pred_test)

print("Validation:", metrics_valid)
print("Test      :", metrics_test)

Validation: {'mae': 5.694672584533691, 'rmse': 31.752450942993164, 'mape': np.float64(13.281017719065897)}
Test      : {'mae': 3.1844332218170166, 'rmse': 23.17032241821289, 'mape': np.float64(2.8981682288587045)}


In [82]:
model.score(x_test, y_test)*100

59.32278633117676

In [83]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.integration import XGBoostPruningCallback

In [84]:
import copy
import pprint

In [85]:
SEED = 42

In [86]:
def create_study(n_trials=50, timeout=None, seed=SEED):
    """Create an Optuna study with a sensible sampler and pruner."""
    sampler = TPESampler(seed=seed)
    pruner = MedianPruner(n_warmup_steps=5)
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    return study

In [87]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 3000, step=50),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_loguniform("gamma", 1e-8, 10.0),

        "tree_method": "hist",
        "random_state": SEED,
        "verbosity": 0,
        "n_jobs": -1,
    }

    model2 = XGBRegressor(**params)

    try:
        model2.fit(
            x_train, y_train,
            eval_set=[(x_valid, y_valid)],
            eval_metric="mae",
            early_stopping_rounds=50,
            verbose=False,
            callbacks=[XGBoostPruningCallback(trial, "validation_0-mae")],
        )
    except Exception as e:
        trial.set_user_attr("error", str(e))
        return float("inf")

    preds = model2.predict(x_valid)
    mae = mean_absolute_error(y_valid, preds)

    return mae


In [88]:
n_trials = 100   # 500 is too heavy unless GPU available
study = create_study(n_trials=n_trials)

print("Starting Optuna study with", n_trials, "trials...")
study.optimize(objective, n_trials=n_trials)

print("\nBest study results:")
print("  Best MAE (validation): {:.4f}".format(study.best_value))
print("  Best trial number:", study.best_trial.number)
print("  Best params:")
pprint.pprint(study.best_params)


[I 2025-11-23 18:01:05,793] A new study created in memory with name: no-name-43447124-f1ee-4b69-84a0-4a71f8bd7227
C:\Users\Dell\AppData\Local\Temp\ipykernel_2696\534971146.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 0.3),
C:\Users\Dell\AppData\Local\Temp\ipykernel_2696\534971146.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
C:\Users\Dell\AppData\Local\Temp\ipykernel_2696\534971146.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/re

Starting Optuna study with 100 trials...


[I 2025-11-23 18:01:05,806] Trial 3 finished with value: inf and parameters: {'n_estimators': 1600, 'learning_rate': 0.011478821143227784, 'max_depth': 3, 'subsample': 0.764526911140863, 'colsample_bytree': 0.502314474212375, 'reg_alpha': 3.850031979199519e-08, 'reg_lambda': 3.4671276804481113, 'min_child_weight': 10, 'gamma': 0.18861495878553936}. Best is trial 0 with value: inf.
[I 2025-11-23 18:01:05,809] Trial 4 finished with value: inf and parameters: {'n_estimators': 950, 'learning_rate': 0.0002185837053086611, 'max_depth': 8, 'subsample': 0.6640914962437607, 'colsample_bytree': 0.47322294090686734, 'reg_alpha': 0.00028614897264046574, 'reg_lambda': 2.039373116525212e-08, 'min_child_weight': 10, 'gamma': 2.133142332373004e-06}. Best is trial 0 with value: inf.
[I 2025-11-23 18:01:05,812] Trial 5 finished with value: inf and parameters: {'n_estimators': 2050, 'learning_rate': 0.0012129899650425088, 'max_depth': 7, 'subsample': 0.7280261676059678, 'colsample_bytree': 0.510912673315


Best study results:
  Best MAE (validation): inf
  Best trial number: 0
  Best params:
{'colsample_bytree': 0.4936111842654619,
 'gamma': 0.002570603566117598,
 'learning_rate': 0.20218499516556748,
 'max_depth': 8,
 'min_child_weight': 9,
 'n_estimators': 1200,
 'reg_alpha': 2.5348407664333426e-07,
 'reg_lambda': 3.3323645788192616e-08,
 'subsample': 0.759195090518222}
